# Day 4: The missing store

**The story.** Harbourside's data now lives in a **warehouse**: one big, central store of tables that the whole company asks questions of. Your manager says:

> "Data's in the warehouse now. Answer these **five questions** before lunch:
> 1. What were our 10 biggest receipts?
> 2. How many sales were from Guests (no loyalty card)?
> 3. What were our busiest days?
> 4. Which product category brings in the most money?
> 5. Which **type** of shop makes the most money: Flagship, High Footfall or Residential?"

To ask a warehouse a question you use **SQL** (say "sequel" or "S-Q-L"): the language almost every company's data speaks. Same ideas as pandas and R, written as an English-like sentence.

**Today works like yesterday.** Your instructor explains each step on the slides and runs it on screen; you run the same cell after them. **Seven** times you change one word. A line under each says **Correct.** or **Not yet.**

| Word | What it means |
| --- | --- |
| **warehouse** | The company's central store of tables. Today it has four: `sales`, `stores`, `customers`, `products` |
| **SQL** | The language for asking a warehouse a question |
| **query** | One question written in SQL |
| `%%sql` | The first line of a SQL cell. Everything under it is pure SQL, sent to the warehouse; the answer comes back as a table |
| `%%sql result <<` | Same, but save the answer as `result` so the next cell can check it |
| **join** | Match two tables on a shared column, like matching two guest lists by name |

In [ ]:
# @title 📦 Step 0 (Optional - For Later): Day 4 SQL Course Pack { display-mode: "form" }
unlock_course_pack = "" # @param {type:"string", placeholder:"add gmail details here"}

import json, re, urllib.request, urllib.parse
from IPython.display import display, HTML
from google.colab import output

_FORM_URL = "https://docs.google.com/forms/d/1qPhzSVDf0V82FKGn-skcemhuQ9y2O1xo4nBgS8P6m8I/viewform"

def _log(action):
    email = unlock_course_pack.strip()
    if not email:
        return
    try:
        resp = urllib.request.urlopen(_FORM_URL, timeout=4)
        final_url = resp.geturl()
        html = resp.read().decode("utf-8", errors="ignore")

        post_match = re.search(r'action="(https://docs\.google\.com/forms/[^"]+/formResponse)"', html)
        post_url = post_match.group(1) if post_match else re.sub(r'/viewform.*$', '/formResponse', final_url)

        fb_match = re.search(r'FB_PUBLIC_LOAD_DATA_\s*=\s*(\[.*?\]);\s*</script>', html, re.DOTALL)
        if fb_match:
            fb_data = json.loads(fb_match.group(1))
            items = fb_data[1][1] if (len(fb_data) > 1 and fb_data[1] and len(fb_data[1]) > 1) else []
            for item in items:
                if item and len(item) > 4 and item[4] and item[4][0]:
                    entry_id = item[4][0][0]
                    value = f"{email} [{action}]"
                    data = urllib.parse.urlencode({f"entry.{entry_id}": value}).encode("utf-8")
                    req = urllib.request.Request(post_url, data=data, headers={"User-Agent": "Mozilla/5.0"})
                    urllib.request.urlopen(req, timeout=4)
                    break
    except Exception:
        pass

output.register_callback('adv.dl', _log)

if unlock_course_pack.strip() and "@" in unlock_course_pack:
    _log("D4_Unlocked_Pack")
    display(HTML("""
    <div style="font-family:sans-serif; font-size:14px; padding:10px; border:1px solid #ccc; border-radius:6px;">
      <b>✅ Day 4 (SQL) Course Pack Unlocked:</b>
      <ul style="margin:6px 0 0 0;">
        <li><a href="https://drive.google.com/drive/folders/1RqQk-uJRbbqauotNLze9S18xNX6ZQ7wc" target="_blank" onclick="google.colab.kernel.invokeFunction('adv.dl', ['D4_Folder_All'], {})">Full Day 4 (SQL) Course Pack Folder</a></li>
        <li><a href="https://docs.google.com/document/d/14yDc_h8Yfha5W77hAJ15XVdvJ9GHnSBojjDdGtP3sKc/export?format=pdf" target="_blank" onclick="google.colab.kernel.invokeFunction('adv.dl', ['D4_Guide_Doc'], {})">Day 4 Guide / Notes (PDF Download)</a></li>
        <li><a href="https://drive.google.com/uc?export=download&id=1DDeXUs5TbRtXeigEsLEPWubO0Uefiwfu" target="_blank" onclick="google.colab.kernel.invokeFunction('adv.dl', ['D4_01_Talk_To_The_Warehouse'], {})">1. Talk to the warehouse</a></li>
        <li><a href="https://drive.google.com/uc?export=download&id=1zqqMj44R7oyHw3ISRQGYQZiXYWfNMdJb" target="_blank" onclick="google.colab.kernel.invokeFunction('adv.dl', ['D4_02_Piles_And_Totals'], {})">2. Piles and totals</a></li>
        <li><a href="https://drive.google.com/uc?export=download&id=1zocT23Hh4DJOnj7ipsLZxD0DLdtmo0AC" target="_blank" onclick="google.colab.kernel.invokeFunction('adv.dl', ['D4_03_The_Missing_Store'], {})">3. The missing store</a></li>
        <li><a href="https://drive.google.com/uc?export=download&id=1o8rFpCtjicXHjFm9EA6IG8orQoVhygmR" target="_blank" onclick="google.colab.kernel.invokeFunction('adv.dl', ['D4_04_Three_Tables_One_Answer'], {})">4. Three tables, one answer</a></li>
      </ul>
    </div>
    """))

## Start here

**Run this.** It builds a small warehouse inside your notebook, with Harbourside's four tables, and loads the same four files into pandas so we can compare. You should see `Ready.`

In [ ]:
%pip install -q jupysql       # the helper that lets a cell speak pure SQL (about 20 seconds; ignore any 'restart' note)
import urllib.request
import duckdb                 # a small SQL warehouse that runs inside the notebook
import pandas as pd

# Where the Harbourside files live (Google Drive). Nothing to change.
DRIVE_IDS = {
    "sales.csv":     "1T0LVklThkbN21YKasuTwl6Hlhtj0uHiK",
    "stores.csv":    "1bZWQiI65hY2ElzONA3VuPFT4omblHtdX",
    "customers.csv": "107sNmG2NYOBdAOUOz0Uc4nbVF84mnlTl",
    "products.csv":  "1S-iZ2vWiRWbOp8Mb9ih63onfJ62B3rOa",
}
def data_url(filename):
    return "https://drive.google.com/uc?export=download&id=" + DRIVE_IDS[filename]

db = duckdb.connect()
for name in ["sales", "stores", "customers", "products"]:
    urllib.request.urlretrieve(data_url(name + ".csv"), name + ".csv")                          # download the file
    db.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM read_csv('{name}.csv')")     # put it in the warehouse

# connect %%sql cells to the warehouse
%load_ext sql
%sql db
%config SqlMagic.displaylimit = 20
%config SqlMagic.feedback = 0

def sql(query):
    return db.sql(query).df()

def as_table(answer):
    # an answer saved with  %%sql result <<  becomes a pandas table, so we can check it
    return answer if isinstance(answer, pd.DataFrame) else answer.DataFrame()

def check(ok, message="Check your answer."):
    print("Correct." if ok else "Not yet. " + message)

# the same four tables in pandas, for comparing with Day 2's language
sales, stores = pd.read_csv("sales.csv"), pd.read_csv("stores.csv")
customers, products = pd.read_csv("customers.csv"), pd.read_csv("products.csv")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 200)
print("Ready. The warehouse has 4 tables: sales, stores, customers, products.")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
Ready. The warehouse has 4 tables: sales, stores, customers, products.


---
# Part 1: Talk to the warehouse

**What we want:** see what's inside a table. Every SQL question has the same shape:

| SQL | In plain English | pandas (Day 2) |
| --- | --- | --- |
| `SELECT *` | Show me every column | `sales` |
| `FROM sales` | from the sales table | |
| `LIMIT 5` | but only the first 5 rows | `.head(5)` |

`*` means "everything". SQL doesn't care about capitals or line breaks; we write the keywords in CAPITALS so they stand out.

**Run this.** Ask the warehouse for the first 5 receipts.

In [ ]:
%%sql
SELECT *
FROM sales
LIMIT 5

transaction_id,date,store_id,customer_id,product_id,quantity,spend_hkd,dwell_time_mins
T00001,2026-09-01,S03,C0480,P16,1,289,21
T00002,2026-09-01,S03,C0415,P23,1,189,15
T00003,2026-09-01,S02,C0139,P21,4,800,25
T00004,2026-09-01,S03,C0251,P13,1,1590,14
T00005,2026-09-01,S01,C0351,P26,5,445,10


**Output**

```
+----------------+------------+----------+-------------+------------+----------+-----------+-----------------+
| transaction_id |    date    | store_id | customer_id | product_id | quantity | spend_hkd | dwell_time_mins |
+----------------+------------+----------+-------------+------------+----------+-----------+-----------------+
|     T00001     | 2026-09-01 |   S03    |    C0480    |    P16     |    1     |    289    |        21       |
|     T00002     | 2026-09-01 |   S03    |    C0415    |    P23     |    1     |    189    |        15       |
|     T00003     | 2026-09-01 |   S02    |    C0139    |    P21     |    4     |    800    |        25       |
|     T00004     | 2026-09-01 |   S03    |    C0251    |    P13     |    1     |    1590   |        14       |
|     T00005     | 2026-09-01 |   S01    |    C0351    |    P26     |    5     |    445    |        10       |
+----------------+------------+----------+-------------+------------+----------+-----------+-----------------+
```

**What we want:** the store list. Only 8 rows, so no LIMIT needed.

**Run this.**

In [ ]:
%%sql
SELECT * FROM stores

store_id,store_name,district,store_type
S01,Harbourside Tsim Sha Tsui,Yau Tsim Mong,Flagship
S02,Harbourside Causeway Bay,Wan Chai,Flagship
S03,Harbourside Mong Kok,Yau Tsim Mong,High Footfall
S04,Harbourside Central,Central and Western,High Footfall
S05,Harbourside Sha Tin,Sha Tin,Residential
S06,Harbourside Tseung Kwan O,Sai Kung,Residential
S07,Harbourside Tuen Mun,Tuen Mun,Residential
S08,Harbourside Kwun Tong,Kwun Tong,High Footfall


**Output**

```
+----------+---------------------------+---------------------+---------------+
| store_id |         store_name        |       district      |   store_type  |
+----------+---------------------------+---------------------+---------------+
|   S01    | Harbourside Tsim Sha Tsui |    Yau Tsim Mong    |    Flagship   |
|   S02    |  Harbourside Causeway Bay |       Wan Chai      |    Flagship   |
|   S03    |    Harbourside Mong Kok   |    Yau Tsim Mong    | High Footfall |
|   S04    |    Harbourside Central    | Central and Western | High Footfall |
|   S05    |    Harbourside Sha Tin    |       Sha Tin       |  Residential  |
|   S06    | Harbourside Tseung Kwan O |       Sai Kung      |  Residential  |
|   S07    |    Harbourside Tuen Mun   |       Tuen Mun      |  Residential  |
|   S08    |   Harbourside Kwun Tong   |      Kwun Tong      | High Footfall |
+----------+---------------------------+---------------------+---------------+
```



In [ ]:
#SO0 is missing

**Your turn 1.** Look inside the **products** table instead. Change **one word**: `stores` to `products`. How many products does Harbourside sell?

**You should see**

```
  product_id       product_name     category  unit_price_hkd
0        P01   Wireless Earbuds  Electronics             890
1        P02  Bluetooth Speaker  Electronics             690
2        P03         Smartwatch  Electronics            2290
Rows: 30
Correct.
```

In [ ]:
%%sql result <<
SELECT * FROM stores   -- Your turn 1: change stores to products

In [ ]:
%%sql result <<
SELECT * FROM products

In [ ]:
result = as_table(result)
print(result.head(3).to_string())
print("Rows:", len(result))
check(len(result) == 30, "Change stores to products, then run again.")

  product_id       product_name     category  unit_price_hkd
0        P01   Wireless Earbuds  Electronics             890
1        P02  Bluetooth Speaker  Electronics             690
2        P03         Smartwatch  Electronics            2290
Rows: 30
Correct.


**What we want:** how big is the sales table, without looking at every row? SQL has a **family of counting buttons**:

| SQL | Question | pandas |
| --- | --- | --- |
| `COUNT(*)` | How many rows? | `len(sales)` |
| `COUNT(customer_id)` | How many rows **have** a card number (empty boxes skipped)? | `sales["customer_id"].count()` |
| `COUNT(DISTINCT store_id)` | How many **different** shops? | `len(sales["store_id"].drop_duplicates())` |
| `MIN(date)`, `MAX(date)` | First and last day | `.min()`, `.max()` |

`AS receipts` gives the answer a name, like a column heading.

In [ ]:
# distinct --> unique

**Run this.** Six counting questions in one query.

In [ ]:
%%sql
SELECT COUNT(*)                    AS receipts,
       COUNT(customer_id)          AS with_card,
       COUNT(DISTINCT customer_id) AS different_members,
       COUNT(DISTINCT store_id)    AS shops,
       MIN(date)                   AS first_day,
       MAX(date)                   AS last_day
FROM sales

# count(*): count rows

receipts,with_card,different_members,shops,first_day,last_day
3583,3140,574,9,2026-09-01,2026-12-31


**Output**

```
+----------+-----------+-------------------+-------+------------+------------+
| receipts | with_card | different_members | shops | first_day  |  last_day  |
+----------+-----------+-------------------+-------+------------+------------+
|   3583   |    3140   |        574        |   9   | 2026-09-01 | 2026-12-31 |
+----------+-----------+-------------------+-------+------------+------------+
```

**Notice two things.**
1. 3,583 receipts, but only 3,140 **with a card**. `COUNT(column)` quietly skips empty boxes. The difference is our Guests.
2. **9 shops**. The store list you just saw has 8. Remember that.

**Same question in pandas.** Same question, in pandas (Day 2's language). Same answer, different spelling.

In [ ]:
print("receipts:         ", len(sales))
print("with_card:        ", sales["customer_id"].count())
print("shops:            ", len(sales["store_id"].drop_duplicates()))
print("first_day:        ", sales["date"].min(), " last_day:", sales["date"].max())

receipts:          3583
with_card:         3140
shops:             9
first_day:         2026-09-01  last_day: 2026-12-31


**Output**

```
receipts:          3583
with_card:         3140
shops:             9
first_day:         2026-09-01  last_day: 2026-12-31
```

**Question 1: our 10 biggest receipts.** `ORDER BY` sorts; `DESC` means biggest first.

| SQL | pandas |
| --- | --- |
| `ORDER BY spend_hkd DESC` | `.sort_values("spend_hkd", ascending=False)` |
| `LIMIT 10` | `.head(10)` |

We also pick only the columns we need, instead of `*`.

**Run this.**

In [ ]:
%%sql
SELECT transaction_id, date, store_id, product_id, quantity, spend_hkd
FROM sales
ORDER BY spend_hkd DESC
LIMIT 10

transaction_id,date,store_id,product_id,quantity,spend_hkd
T00122,2026-09-06,S07,P06,2,15980
T02580,2026-12-02,S07,P06,2,15980
T02880,2026-12-13,S05,P06,2,15980
T00200,2026-09-09,S06,P05,2,9180
T01334,2026-10-14,S08,P05,2,9180
T01636,2026-10-25,S02,P05,2,9180
T02198,2026-11-17,S03,P05,2,9180
T02864,2026-12-13,S01,P05,2,9180
T00268,2026-09-11,S04,P06,1,7990
T00526,2026-09-20,S06,P06,1,7990


**Output**

```
+----------------+------------+----------+------------+----------+-----------+
| transaction_id |    date    | store_id | product_id | quantity | spend_hkd |
+----------------+------------+----------+------------+----------+-----------+
|     T00122     | 2026-09-06 |   S07    |    P06     |    2     |   15980   |
|     T02580     | 2026-12-02 |   S07    |    P06     |    2     |   15980   |
|     T02880     | 2026-12-13 |   S05    |    P06     |    2     |   15980   |
|     T00200     | 2026-09-09 |   S06    |    P05     |    2     |    9180   |
|     T01334     | 2026-10-14 |   S08    |    P05     |    2     |    9180   |
|     T01636     | 2026-10-25 |   S02    |    P05     |    2     |    9180   |
|     T02198     | 2026-11-17 |   S03    |    P05     |    2     |    9180   |
|     T02864     | 2026-12-13 |   S01    |    P05     |    2     |    9180   |
|     T00268     | 2026-09-11 |   S04    |    P06     |    1     |    7990   |
|     T00526     | 2026-09-20 |   S06    |    P06     |    1     |    7990   |
+----------------+------------+----------+------------+----------+-----------+
```

**Answer 1:** the biggest receipt is **HK\$15,980** (three times: two laptops each, product P06). All ten are Electronics products (P05, P06). ✅ One question down.

**Same question in pandas.** Same question, in pandas (Day 2's language). Same answer, different spelling.

In [ ]:
print(sales.sort_values("spend_hkd", ascending=False)
           [["transaction_id", "date", "store_id", "product_id", "quantity", "spend_hkd"]]
           .head(10).to_string(index=False))

transaction_id       date store_id product_id  quantity  spend_hkd
        T02580 2026-12-02      S07        P06         2      15980
        T02880 2026-12-13      S05        P06         2      15980
        T00122 2026-09-06      S07        P06         2      15980
        T01334 2026-10-14      S08        P05         2       9180
        T01636 2026-10-25      S02        P05         2       9180
        T02198 2026-11-17      S03        P05         2       9180
        T00200 2026-09-09      S06        P05         2       9180
        T02864 2026-12-13      S01        P05         2       9180
        T01989 2026-11-08      S02        P06         1       7990
        T01004 2026-10-04      S01        P06         1       7990


**Break it on purpose.** In SQL, **text goes in single quotes** `'S01'`. Double quotes `"S01"` mean "a column called S01". Run this. It will go red.

**You should see**

```
BinderException: Binder Error: Referenced column "S01" not found in FROM clause!
```

In [ ]:
%%sql
SELECT * FROM sales WHERE store_id = "S01" LIMIT 3

BinderException: Binder Error: Referenced column "S01" not found in FROM clause!
Candidate bindings: "date"

LINE 1: SELECT * FROM sales WHERE store_id = "S01" LIMIT 3
                                             ^

**How to read it:** `column "S01" not found`. SQL went looking for a **column** called S01, because of the double quotes. Fix: `'S01'`. Your Python and R habits (double quotes) will bite here all day; that's normal.

---
# Part 2: Keep only some rows

**What we want:** ask about some receipts, not all. `WHERE` keeps only the rows that pass a test.

| SQL | pandas | R (Day 3) |
| --- | --- | --- |
| `WHERE store_id = 'S01'` | `sales[sales["store_id"] == "S01"]` | `filter(store_id == "S01")` |

In SQL, **one** `=` asks "is it equal?". (In Python and R that's `==`.) Different languages, different spelling.

**Run this.** How many receipts at Tsim Sha Tsui (S01)?

In [ ]:
%%sql
SELECT COUNT(*) AS receipts, SUM(spend_hkd) AS total
FROM sales
WHERE store_id = 'S01'

receipts,total
571,639275


**Output**

```
+----------+--------+
| receipts | total  |
+----------+--------+
|   571    | 639275 |
+----------+--------+
```

**Your turn 2.** Ask about the mystery shop **S09** instead. Change **one word**: `S01` to `S09` (keep the single quotes).

In [ ]:
%%sql
SELECT COUNT(*) AS receipts, SUM(spend_hkd) AS total
FROM sales
WHERE store_id = 'S09'

receipts,total
285,322939


**You should see**

```
 receipts      total
      285 322,939.00
Correct.
```

In [ ]:
%%sql result <<
SELECT COUNT(*) AS receipts, SUM(spend_hkd) AS total FROM sales WHERE store_id = 'S01'   -- Your turn 2: change S01 to S09

In [ ]:
result = as_table(result)
print(result.to_string(index=False))
check(result["receipts"][0] == 285, "Change S01 to S09, keep the single quotes, then run again.")

S09 took **HK\$322,939**: more than Tuen Mun, a whole shop's worth. And it isn't on the store list. Keep that number in your head until Part 4.

**More than one test.** `AND` = both must be true. `OR` = at least one. `IN (...)` = "is one of". `BETWEEN` = "from ... to ...".

| SQL | pandas |
| --- | --- |
| `WHERE store_id = 'S01' AND spend_hkd >= 1000` | `sales[(sales["store_id"] == "S01") & (sales["spend_hkd"] >= 1000)]` |
| `WHERE store_id IN ('S01', 'S02')` | *New today. Not in yesterday's Python.* |
| `WHERE date BETWEEN '2026-12-18' AND '2026-12-24'` | *New today. Not in yesterday's Python.* |

**Run this.** Three filtered counts in one go: the two flagships' big receipts, and Christmas week.

In [ ]:
%%sql
SELECT
  (SELECT COUNT(*) FROM sales WHERE store_id IN ('S01', 'S02') AND spend_hkd > 5000)       AS flagship_big_receipts,
  (SELECT COUNT(*) FROM sales WHERE date BETWEEN '2026-12-18' AND '2026-12-24')              AS christmas_week_receipts,
  (SELECT COUNT(*) FROM sales WHERE date BETWEEN '2026-12-18' AND '2026-12-24') / 7.0        AS per_day

flagship_big_receipts,christmas_week_receipts,per_day
29,432,61.714285714285715


**Output**

```
+-----------------------+-------------------------+--------------------+
| flagship_big_receipts | christmas_week_receipts |      per_day       |
+-----------------------+-------------------------+--------------------+
|           29          |           432           | 61.714285714285715 |
+-----------------------+-------------------------+--------------------+
```

**Your turn 3.** This counts Tsim Sha Tsui receipts of HK\$1,000 or more. Change **one word**: `AND` to `OR`. Predict first: will the number go up or down?

**You should see**

```
   n
1456
Correct.
```

In [ ]:
%%sql result <<
SELECT COUNT(*) AS n FROM sales WHERE store_id = 'S01' OR spend_hkd >= 1000   -- Your turn 3: change AND to OR

In [ ]:
result = as_table(result)
print(result.to_string(index=False))
check(result["n"][0] == 1456, "Change AND to OR, then run again.")

   n
1456
Correct.


In [ ]:
%%sql result <<
SELECT COUNT(*) AS n FROM sales WHERE store_id = 'S01' AND spend_hkd >= 1000   -- Your turn 3: change AND to OR

In [ ]:
result = as_table(result)
print(result.to_string(index=False))
check(result["n"][0] == 163)

  n
163
Correct.


163 with `AND` (Tsim Sha Tsui **and** big). 1,456 with `OR` (Tsim Sha Tsui, **or** big anywhere). One small word, nine times the answer.

**Question 2: how many Guest sales?** A Guest has an **empty** `customer_id`. In SQL an empty box is called `NULL`.

**Wrong turn 1.** The obvious way: ask for rows where customer_id equals NULL.

**Run this.**

In [ ]:
%%sql
SELECT COUNT(*) AS guests FROM sales WHERE customer_id = NULL

guests
0


**Output**

```
+--------+
| guests |
+--------+
|   0    |
+--------+
```

**Zero?** But we know 3,583 − 3,140 = 443 receipts have no card. **No red text, and a wrong answer.** That's the most dangerous kind of mistake.

**Why: NULL means "unknown".** Is an unknown equal to an unknown? SQL says "I can't tell", so no row passes. To ask "is this box empty?", SQL has special words: `IS NULL` and `IS NOT NULL`.

**Run this.** The right way.

In [ ]:
%%sql
SELECT COUNT(*) AS guests,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM sales), 1) AS percent_of_receipts,
       SUM(spend_hkd) AS guest_spend
FROM sales
WHERE customer_id IS NULL

guests,percent_of_receipts,guest_spend
443,12.4,484336


**Output**

```
+--------+---------------------+-------------+
| guests | percent_of_receipts | guest_spend |
+--------+---------------------+-------------+
|  443   |         12.4        |    484336   |
+--------+---------------------+-------------+
```

**Answer 2:** **443 Guest sales**, 12.4% of receipts, worth HK\$484,336. Same 443 as Day 2. ✅

**Same question in pandas.** pandas has its own word for the same idea: `.isna()`.

In [ ]:
print("guests:", sales["customer_id"].isna().sum())

**Output**

```
guests: 443
```

---
# Part 3: Piles and totals

**What we want:** add things up. SQL's **summary buttons** are the same family you met on Day 3:

| SQL | pandas | R |
| --- | --- | --- |
| `SUM(spend_hkd)` | `.sum()` | `sum()` |
| `COUNT(*)` | `len()` / `.count()` | `n()` |
| `AVG(spend_hkd)` | `.mean()` | `mean()` |
| `MEDIAN(spend_hkd)` | *not in our Python* | `median()` |
| `MIN(...)`, `MAX(...)` | `.min()`, `.max()` | `min()`, `max()` |

**Run this.** All of them at once, on every receipt.

In [ ]:
%%sql
SELECT SUM(spend_hkd)           AS total,
       COUNT(*)                 AS receipts,
       ROUND(AVG(spend_hkd), 2) AS average,
       MEDIAN(spend_hkd)        AS middle,
       MIN(spend_hkd)           AS smallest,
       MAX(spend_hkd)           AS biggest
FROM sales

total,receipts,average,middle,smallest,biggest
3904460,3583,1089.72,578.0,89,15980


**Output**

```
+---------+----------+---------+--------+----------+---------+
|  total  | receipts | average | middle | smallest | biggest |
+---------+----------+---------+--------+----------+---------+
| 3904460 |   3583   | 1089.72 | 578.0  |    89    |  15980  |
+---------+----------+---------+--------+----------+---------+
```

The same numbers as yesterday in R: total **HK\$3,904,460**, average HK\$1,089.72, middle HK\$578. Three languages, one truth.

**What we want:** one row per shop. `GROUP BY` sorts the receipts into piles first, then the buttons work once per pile.

| SQL | pandas | R |
| --- | --- | --- |
| `GROUP BY store_id` | `.groupby("store_id")` | `group_by(store_id)` |

**Run this.** One pile per shop, three answers per pile, biggest total first.

In [ ]:
%%sql
SELECT store_id,
       COUNT(*)                 AS receipts,
       SUM(spend_hkd)           AS total,
       ROUND(AVG(spend_hkd), 2) AS average
FROM sales
GROUP BY store_id
ORDER BY total DESC

store_id,receipts,total,average
S01,571,639275,1119.57
S02,518,563637,1088.1
S03,464,475758,1025.34
S04,408,459648,1126.59
S05,386,404010,1046.66
S08,330,375297,1137.26
S06,319,350105,1097.51
S09,285,322939,1133.12
S07,302,313791,1039.04


**Output**

```
+----------+----------+--------+---------+
| store_id | receipts | total  | average |
+----------+----------+--------+---------+
|   S01    |   571    | 639275 | 1119.57 |
|   S02    |   518    | 563637 |  1088.1 |
|   S03    |   464    | 475758 | 1025.34 |
|   S04    |   408    | 459648 | 1126.59 |
|   S05    |   386    | 404010 | 1046.66 |
|   S08    |   330    | 375297 | 1137.26 |
|   S06    |   319    | 350105 | 1097.51 |
|   S09    |   285    | 322939 | 1133.12 |
|   S07    |   302    | 313791 | 1039.04 |
+----------+----------+--------+---------+
```

**Your turn 4.** Which shop has the biggest **average** receipt? Change **one word**: `SUM` to `AVG`.

**You should see**

```
store_id    value
     S08 1,137.26
     S09 1,133.12
     S04 1,126.59
Correct.
```

In [ ]:
%%sql result <<
SELECT store_id, ROUND(SUM(spend_hkd), 2) AS value      -- Your turn 4: change SUM to AVG
FROM sales
GROUP BY store_id
ORDER BY value DESC
LIMIT 3

In [ ]:
result = as_table(result)
print(result.to_string(index=False))
check(result["store_id"][0] == "S08", "Change SUM to AVG, then run again.")

S08 Kwun Tong, as in R yesterday. And S09, the shop that "doesn't exist", is second. `--` starts a **comment** in SQL: a note for humans that the warehouse ignores (Python and R use `#`).

**Same question in pandas.** Same question, in pandas (Day 2's language). Same answer, different spelling.

In [ ]:
print(sales.groupby("store_id")["spend_hkd"].mean().round(2).sort_values(ascending=False).head(3).to_string())

**Output**

```
store_id
S08   1,137.26
S09   1,133.12
S04   1,126.59
```

**Break it on purpose.** Every column in `SELECT` must either be a pile label (in `GROUP BY`) or be inside a button like `SUM`. Here `store_id` is neither. Run it; it will go red.

**You should see**

```
BinderException: Binder Error: column "store_id" must appear in the GROUP BY clause or must be part of an aggregate function.
```

In [ ]:
%%sql
SELECT store_id, SUM(spend_hkd) AS total FROM sales
GROUP BY store_id
#注意那个group by

store_id,total
S05,404010
S04,459648
S08,375297
S03,475758
S02,563637
S01,639275
S07,313791
S06,350105
S09,322939


In English: "You asked for each `store_id` **and** one grand total. Which do you want?" Fix: add `GROUP BY store_id`.

**Question 3: our busiest days.** Pile up by `date`. `dayname(date)` turns a date into Monday, Tuesday...

**Run this.** Top 5 days by money.

In [ ]:
%%sql
SELECT date,
       dayname(date)  AS weekday,
       COUNT(*)       AS receipts,
       SUM(spend_hkd) AS total
FROM sales
GROUP BY date
ORDER BY total DESC
LIMIT 5

date,weekday,receipts,total
2026-12-19,Saturday,79,110978
2026-10-03,Saturday,85,97486
2026-12-20,Sunday,78,93781
2026-10-04,Sunday,64,77539
2026-12-18,Friday,71,75697


**Output**

```
+------------+----------+----------+--------+
|    date    | weekday  | receipts | total  |
+------------+----------+----------+--------+
| 2026-12-19 | Saturday |    79    | 110978 |
| 2026-10-03 | Saturday |    85    | 97486  |
| 2026-12-20 |  Sunday  |    78    | 93781  |
| 2026-10-04 |  Sunday  |    64    | 77539  |
| 2026-12-18 |  Friday  |    71    | 75697  |
+------------+----------+----------+--------+
```

**Your turn 5.** "Busiest" could mean most **shoppers** instead of most money. Change **one word** after `ORDER BY`: `total` to `receipts`.

**You should see**

```
      date  weekday  receipts      total
2026-10-03 Saturday        85  97,486.00
2026-12-19 Saturday        79 110,978.00
2026-12-20   Sunday        78  93,781.00
2026-12-18   Friday        71  75,697.00
2026-10-04   Sunday        64  77,539.00
Correct.
```

In [ ]:
%%sql result <<
SELECT date, dayname(date) AS weekday, COUNT(*) AS receipts, SUM(spend_hkd) AS total
FROM sales
GROUP BY date
ORDER BY total DESC          -- Your turn 5: change total to receipts
LIMIT 5

In [ ]:
result = as_table(result)
print(result.to_string(index=False))
check(str(result["date"][0])[:10] == "2026-10-03", "Change total to receipts after ORDER BY, then run again.")

      date  weekday  receipts      total
2026-12-19 Saturday        79 110,978.00
2026-10-03 Saturday        85  97,486.00
2026-12-20   Sunday        78  93,781.00
2026-10-04   Sunday        64  77,539.00
2026-12-18   Friday        71  75,697.00
Not yet. Change total to receipts after ORDER BY, then run again.


**Answer 3:** by money, **Saturday 19 December** (HK\$110,978, the Christmas run-up). By shoppers, **Saturday 3 October** (85 receipts, Golden Week). Both are right; they answer different questions. Ask the manager which one she meant. ✅

**Filtering piles: HAVING.** `WHERE` filters **receipts** before they're piled up. `HAVING` filters **piles** after they're added up. "Show me only days over HK\$70,000":

**Run this.**

In [ ]:
%%sql
SELECT date, dayname(date) AS weekday, SUM(spend_hkd) AS total
FROM sales
GROUP BY date
HAVING SUM(spend_hkd) > 70000
ORDER BY total DESC

date,weekday,total
2026-12-19,Saturday,110978
2026-10-03,Saturday,97486
2026-12-20,Sunday,93781
2026-10-04,Sunday,77539
2026-12-18,Friday,75697


**Output**

```
+------------+----------+--------+
|    date    | weekday  | total  |
+------------+----------+--------+
| 2026-12-19 | Saturday | 110978 |
| 2026-10-03 | Saturday | 97486  |
| 2026-12-20 |  Sunday  | 93781  |
| 2026-10-04 |  Sunday  | 77539  |
| 2026-12-18 |  Friday  | 75697  |
+------------+----------+--------+
```

Only 5 days in four months took more than HK\$70,000: two in Golden Week, three just before Christmas. A good fact for staffing.

### If we have time: which day of the week?

**Run this.**

In [ ]:
%%sql
SELECT dayname(date) AS weekday, COUNT(*) AS receipts, ROUND(AVG(spend_hkd)) AS average
FROM sales
GROUP BY weekday
ORDER BY receipts DESC

**Output**

```
+-----------+----------+---------+
|  weekday  | receipts | average |
+-----------+----------+---------+
|  Saturday |   727    |  1076.0 |
|   Sunday  |   658    |  1175.0 |
|   Friday  |   458    |  1023.0 |
|  Tuesday  |   454    |  1019.0 |
| Wednesday |   446    |  1173.0 |
|  Thursday |   423    |  1057.0 |
|   Monday  |   417    |  1073.0 |
+-----------+----------+---------+
```

---
# Part 4: The missing store

**What we want:** questions 4 and 5 need information that isn't in `sales`. The category lives in `products`; the shop type lives in `stores`. A **JOIN** matches two tables on a shared column, like matching a guest list to a seating plan by name.

| SQL | pandas | R |
| --- | --- | --- |
| `FROM sales JOIN products ON sales.product_id = products.product_id` | `sales.merge(products, on="product_id")` | `inner_join(sales, products, by = "product_id")` |

`s` and `p` are nicknames (`FROM sales s` means "call sales s"), so we can write `s.spend_hkd` instead of `sales.spend_hkd`.

**Question 4: which category brings in the most money?**

**Run this.**

In [ ]:
%%sql
SELECT p.category,
       COUNT(*)         AS receipts,
       SUM(s.quantity)  AS units,
       SUM(s.spend_hkd) AS total
FROM sales s
JOIN products p ON s.product_id = p.product_id
GROUP BY p.category
ORDER BY total DESC

**Output**

```
+-------------+----------+-------+---------+
|   category  | receipts | units |  total  |
+-------------+----------+-------+---------+
| Electronics |   610    |  684  | 1636234 |
|     Home    |   1215   |  1918 | 1185128 |
|    Beauty   |   1758   |  4222 | 1083098 |
+-------------+----------+-------+---------+
```

**Answer 4:** **Electronics**, HK\$1,636,234, from the fewest units (684). Same as Day 2. Check the three totals add up: 1,636,234 + 1,185,128 + 1,083,098 = **3,904,460**. Every receipt found its product. ✅

**Question 5: which type of shop makes the most money?** Same pattern, joining `stores` this time.

**Wrong turn 2.** Join, pile up by store type, done.

**Run this.**

In [ ]:
%%sql
SELECT st.store_type,
       COUNT(*)         AS receipts,
       SUM(s.spend_hkd) AS total
FROM sales s
JOIN stores st ON s.store_id = st.store_id
GROUP BY st.store_type
ORDER BY total DESC

**Output**

```
+---------------+----------+---------+
|   store_type  | receipts |  total  |
+---------------+----------+---------+
| High Footfall |   1202   | 1310703 |
|    Flagship   |   1089   | 1202912 |
|  Residential  |   1007   | 1067906 |
+---------------+----------+---------+
```

**High Footfall wins.** Write it in the report? Do what we did for Question 4 first: **check the totals add up**.

**Run this.** Add up the three store types, and compare with the real total.

In [ ]:
%%sql by_type <<
SELECT SUM(s.spend_hkd) AS total, COUNT(*) AS receipts
FROM sales s
JOIN stores st ON s.store_id = st.store_id

In [ ]:
by_type = as_table(by_type)
print("Joined total:", f"HK${by_type['total'][0]:,.0f}", " from", by_type["receipts"][0], "receipts")
print("Real total:   HK$3,904,460  from 3583 receipts")
print("Missing:     ", f"HK${3904460 - by_type['total'][0]:,.0f}", " from", 3583 - by_type["receipts"][0], "receipts")

**Output**

```
Joined total: HK$3,581,521  from 3298 receipts
Real total:   HK$3,904,460  from 3583 receipts
Missing:      HK$322,939  from 285 receipts
```

**HK\$322,939 has vanished, with no red text.** 285 receipts... the same 285 as shop S09.

What happened: a plain `JOIN` (an **INNER** join) keeps only receipts that find a match in **both** tables. S09 isn't in `stores`, so its receipts were silently thrown away. A **LEFT JOIN** keeps every receipt from the left table (`sales`), and leaves the store details empty (`NULL`) where there's no match.

**Your turn 6.** Change **one word**: `JOIN` to `LEFT JOIN`. Does the missing money come back?

**You should see**

```
   store_type  receipts        total
High Footfall      1202 1,310,703.00
     Flagship      1089 1,202,912.00
  Residential      1007 1,067,906.00
          NaN       285   322,939.00
Correct.
```

In [ ]:
%%sql result <<
SELECT st.store_type, COUNT(*) AS receipts, SUM(s.spend_hkd) AS total
FROM sales s
JOIN stores st ON s.store_id = st.store_id          -- Your turn 6: change JOIN to LEFT JOIN
GROUP BY st.store_type
ORDER BY total DESC

In [ ]:
result = as_table(result)
print(result.to_string(index=False))
check(result["total"].sum() == 3904460, "Change JOIN to LEFT JOIN, then run again.")

      date  weekday  receipts      total
2026-12-19 Saturday        79 110,978.00
2026-10-03 Saturday        85  97,486.00
2026-12-20   Sunday        78  93,781.00
2026-10-04   Sunday        64  77,539.00
2026-12-18   Friday        71  75,697.00
Not yet. Change JOIN to LEFT JOIN, then run again.


The fourth row (`None` / `NaN`: no store type) holds the missing HK\$322,939. Now the totals add up. So what **is** S09?

**Run this.** When did S09 first sell anything?

In [ ]:
%%sql
SELECT MIN(date) AS first_sale, MAX(date) AS last_sale, COUNT(*) AS receipts, SUM(spend_hkd) AS total
FROM sales
WHERE store_id = 'S09'

first_sale,last_sale,receipts,total
2026-10-20,2026-12-31,285,322939


**Output**

```
+------------+------------+----------+--------+
| first_sale | last_sale  | receipts | total  |
+------------+------------+----------+--------+
| 2026-10-20 | 2026-12-31 |   285    | 322939 |
+------------+------------+----------+--------+
```

**No sales before 20 October.** Every other shop sold from 1 September. S09 is a **new shop**. The manager confirms: it's **Kai Tak**, opened 20 October, and nobody added it to the store list. It's a big new mall shop, so they class it as a **Flagship**.

`CASE WHEN ... THEN ... ELSE ... END` is SQL's **if/else** (Day 1's "if it's raining, take an umbrella"):

**Run this.** Treat S09 as a Flagship, and ask Question 5 again.

In [ ]:
# store_type = NULL
# 把S09当成 Flagship Store

In [ ]:
%%sql
SELECT CASE WHEN s.store_id = 'S09' THEN 'Flagship' ELSE st.store_type END AS store_type,
       COUNT(*)         AS receipts,
       SUM(s.spend_hkd) AS total
FROM sales s
LEFT JOIN stores st ON s.store_id = st.store_id
GROUP BY 1
ORDER BY total DESC

store_type,receipts,total
Flagship,1374,1525851
High Footfall,1202,1310703
Residential,1007,1067906


**Output**

```
+---------------+----------+---------+
|   store_type  | receipts |  total  |
+---------------+----------+---------+
|    Flagship   |   1374   | 1525851 |
| High Footfall |   1202   | 1310703 |
|  Residential  |   1007   | 1067906 |
+---------------+----------+---------+
```

**The answer flips.** With Kai Tak counted, **Flagship** wins: HK\$1,525,851 against High Footfall's HK\$1,310,703, and Kai Tak has only been open 10 weeks.

**Answer 5:** Flagship, **once you count the shop the store list forgot**. The plain JOIN gave the wrong answer, with no error, because of one missing row in a reference table. ✅

**Same question in pandas.** `merge` has the same two kinds: `how="inner"` (the default) and `how="left"`.

In [ ]:
inner = sales.merge(stores, on="store_id", how="inner")
left  = sales.merge(stores, on="store_id", how="left")
print("inner merge:", len(inner), "receipts, HK$", f"{inner['spend_hkd'].sum():,}")
print("left merge: ", len(left),  "receipts, HK$", f"{left['spend_hkd'].sum():,}")

inner merge: 3298 receipts, HK$ 3,581,521
left merge:  3583 receipts, HK$ 3,904,460


**Output**

```
inner merge: 3298 receipts, HK$ 3,581,521
left merge:  3583 receipts, HK$ 3,904,460
```

---
# Part 5: Per what?

**What we want:** check whether "Flagship wins" is fair. There are **3** High Footfall shops and only **2** listed Flagships (3 with Kai Tak). More shops, more money. A fairer question: **money per shop**.

`COUNT(DISTINCT s.store_id)` counts the shops in each pile.

**Run this.** Total and money per shop, for each store type (Kai Tak left as Unknown).

In [ ]:
%%sql
SELECT COALESCE(st.store_type, 'Unknown (S09)')                    AS store_type,
       COUNT(DISTINCT s.store_id)                                  AS shops,
       SUM(s.spend_hkd)                                            AS total,
       ROUND(SUM(s.spend_hkd) / COUNT(DISTINCT s.store_id))        AS per_shop
FROM sales s
LEFT JOIN stores st ON s.store_id = st.store_id
GROUP BY 1
ORDER BY per_shop DESC

store_type,shops,total,per_shop
Flagship,2,1202912,601456.0
High Footfall,3,1310703,436901.0
Residential,3,1067906,355969.0
Unknown (S09),1,322939,322939.0


**Output**

```
+---------------+-------+---------+----------+
|   store_type  | shops |  total  | per_shop |
+---------------+-------+---------+----------+
|    Flagship   |   2   | 1202912 | 601456.0 |
| High Footfall |   3   | 1310703 | 436901.0 |
|  Residential  |   3   | 1067906 | 355969.0 |
| Unknown (S09) |   1   |  322939 | 322939.0 |
+---------------+-------+---------+----------+
```

`COALESCE(x, 'Unknown')` means "use x, or 'Unknown' if x is empty". (pandas: `.fillna("Unknown")`, Day 2.)

**Per shop, Flagships win easily**: about HK\$601,000 each, against HK\$437,000 for a High Footfall shop. And Kai Tak made HK\$322,939 in just 10 weeks. Whatever the question, **always ask "per what?"**: per shop, per receipt, per customer, per day.

**Three tables at once: who are our best customers?** The loyalty tier (Bronze, Silver, Gold) lives in `customers`. Join it in too.

**Run this.** Total spend by loyalty tier.

In [ ]:
%%sql tiers <<
SELECT c.membership_tier,
       COUNT(DISTINCT s.customer_id)                                   AS members,
       SUM(s.spend_hkd)                                                AS total,
       ROUND(SUM(s.spend_hkd) / COUNT(DISTINCT s.customer_id))         AS per_member
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.membership_tier
ORDER BY total DESC

In [ ]:
tiers = as_table(tiers)
print(tiers.to_string(index=False))
print("Total of the tiers:", f"HK${tiers['total'].sum():,.0f}", "  (real total HK$3,904,460)")

membership_tier  members        total  per_member
         Bronze      301 1,424,568.00    4,733.00
         Silver      174 1,063,779.00    6,114.00
           Gold       99   931,777.00    9,412.00
Total of the tiers: HK$3,420,124   (real total HK$3,904,460)


**Output**

```
membership_tier  members        total  per_member
         Bronze      301 1,424,568.00    4,733.00
         Silver      174 1,063,779.00    6,114.00
           Gold       99   931,777.00    9,412.00
Total of the tiers: HK$3,420,124   (real total HK$3,904,460)
```

**Same trap again.** The tiers add up to HK\$3,420,124: **HK\$484,336 short**. Who's missing? The **Guests**: no card, so no match in `customers`, so the plain JOIN dropped them. You know the fix now (LEFT JOIN); we'll leave it, because here we only want members.

**Your turn 7.** Bronze has the biggest **total**, but Bronze also has the most members. Sort by money **per member** instead. Change **one word** after `ORDER BY`: `total` to `per_member`.

**You should see**

```
membership_tier        total  per_member
           Gold   931,777.00    9,412.00
         Silver 1,063,779.00    6,114.00
         Bronze 1,424,568.00    4,733.00
Correct.
```

In [ ]:
%%sql result <<
SELECT c.membership_tier,
       SUM(s.spend_hkd) AS total,
       ROUND(SUM(s.spend_hkd) / COUNT(DISTINCT s.customer_id)) AS per_member
FROM sales s
JOIN customers c ON s.customer_id = c.customer_id
GROUP BY c.membership_tier
ORDER BY total DESC            -- Your turn 7: change total to per_member

In [ ]:
result = as_table(result)
print(result.to_string(index=False))
check(result["membership_tier"][0] == "Gold", "Change total to per_member after ORDER BY, then run again.")

membership_tier        total  per_member
         Bronze 1,424,568.00    4,733.00
         Silver 1,063,779.00    6,114.00
           Gold   931,777.00    9,412.00
Not yet. Change total to per_member after ORDER BY, then run again.


**Flipped again.** Each **Gold** member spends about HK\$9,400 over four months, twice a Bronze member's HK\$4,700. Bronze only wins on total because there are three times as many of them. "Per what?" changed the answer.

**One more SQL idea you'll see at work: `WITH`.** For long questions,analysts build a small named table first, then ask about it.It reads top to bottom, like R's pipe.

**Run this.** Step 1: a table of each member's spend. Step 2: the five biggest spenders, with their tier.

In [ ]:
%%sql
WITH member_spend AS (
    SELECT customer_id, COUNT(*) AS visits, SUM(spend_hkd) AS total
    FROM sales
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
)
#SUM(spend_hkd) AS total, 对临时表提问
SELECT m.customer_id, c.membership_tier, m.visits, m.total
FROM member_spend m
JOIN customers c ON m.customer_id = c.customer_id
ORDER BY m.total DESC
LIMIT 5

customer_id,membership_tier,visits,total
C0278,Bronze,152,178720
C0528,Gold,101,108573
C0480,Silver,83,107882
C0049,Silver,97,102469
C0139,Bronze,84,91646


**Output**

```
+-------------+-----------------+--------+--------+
| customer_id | membership_tier | visits | total  |
+-------------+-----------------+--------+--------+
|    C0278    |      Bronze     |  152   | 178720 |
|    C0528    |       Gold      |  101   | 108573 |
|    C0480    |      Silver     |   83   | 107882 |
|    C0049    |      Silver     |   97   | 102469 |
|    C0139    |      Bronze     |   84   | 91646  |
+-------------+-----------------+--------+--------+
```

---
# Part 6: The five answers

| # | Question | Answer | The trap |
| --- | --- | --- | --- |
| 1 | 10 biggest receipts | HK\$15,980 at the top, all Electronics | None |
| 2 | Guest sales | 443 (12.4%), HK\$484,336 | `= NULL` gives 0 with no error; use `IS NULL` |
| 3 | Busiest days | By money Sat 19 Dec; by shoppers Sat 3 Oct | "Busiest" means two things |
| 4 | Top category | Electronics, HK\$1,636,234 | None, and the total checks |
| 5 | Top store type | Flagship, once Kai Tak (S09) is counted | A plain JOIN silently drops the shop missing from the list |

## What you did today
1. Asked a warehouse questions with `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT`.
2. Pressed SQL's summary buttons (`COUNT`, `SUM`, `AVG`, `MEDIAN`, `MIN`, `MAX`) and piled them up with `GROUP BY`.
3. Took three wrong turns on purpose: `= NULL`, an INNER JOIN that lost a shop, and totals that weren't "per" anything.
4. Saw every big question in SQL **and** pandas: same idea, different spelling.

**If something goes wrong:** read the last line of the red text. Text in **single** quotes. **Runtime > Run all** fixes most things.